# 04. GitHub Pages 대시보드 데이터 생성

**목표:** MySQL에 저장된 분석 결과를 정적 웹페이지에서 사용할 수 있는 JSON 파일로 변환합니다.

생성 파일:
- `market.json` — 2023~2025 자치구별 월간 임대차 시장 데이터
- `vacancy.json` — 2023~2025 자치구별 연간 빈집 분석 데이터
- `forecast.json` — 2026년 1월 자치구별 임대차 거래량 예측
- `summary.json` — 메인 화면용 요약 정보


## 1. MySQL 분석 결과 로드


In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

market = pd.read_sql("SELECT * FROM ml_rent_market", engine)
vacancy = pd.read_sql("SELECT * FROM vacancy_market_analysis", engine)
forecast = pd.read_sql("SELECT * FROM rent_forecast_2026_01", engine)

print("시장 데이터:", market.shape)
print("빈집 데이터:", vacancy.shape)
print("2026-01 예측 데이터:", forecast.shape)


시장 데이터: (900, 8)
빈집 데이터: (75, 16)
2026-01 예측 데이터: (25, 4)


## 2. 웹용 JSON 생성


In [2]:
output_dir = "../docs/data"
os.makedirs(output_dir, exist_ok=True)

market.to_json(
    f"{output_dir}/market.json",
    orient="records",
    force_ascii=False
)

vacancy.to_json(
    f"{output_dir}/vacancy.json",
    orient="records",
    force_ascii=False
)

forecast.to_json(
    f"{output_dir}/forecast.json",
    orient="records",
    force_ascii=False
)

print("market.json 저장:", len(market))
print("vacancy.json 저장:", len(vacancy))
print("forecast.json 저장:", len(forecast))


market.json 저장: 900
vacancy.json 저장: 75
forecast.json 저장: 25


## 3. 메인 화면 요약 데이터 생성


In [3]:
summary = {
    "years": [2023, 2024, 2025],
    "districtCount": int(market["guName"].nunique()),
    "vacancyTotal": {
        str(year): int(
            vacancy[vacancy["year"] == year]["total"].sum()
        )
        for year in [2023, 2024, 2025]
    },
    "forecastMonth": "2026-01"
}

pd.Series(summary).to_json(
    f"{output_dir}/summary.json",
    force_ascii=False
)

print(summary)


{'years': [2023, 2024, 2025], 'districtCount': 25, 'vacancyTotal': {'2023': 107681, '2024': 102556, '2025': 126290}, 'forecastMonth': '2026-01'}


## 4. 생성 파일 검증


In [4]:
for filename in [
    "market.json",
    "vacancy.json",
    "forecast.json",
    "summary.json"
]:
    path = f"{output_dir}/{filename}"
    print(
        filename,
        ":",
        os.path.exists(path),
        os.path.getsize(path),
        "bytes"
    )


market.json : True 141459 bytes
vacancy.json : True 23339 bytes
forecast.json : True 2678 bytes
summary.json : True 130 bytes


## 5. 산출물

`docs/data/` 아래의 JSON 파일은 GitHub Pages의 HTML/JavaScript에서 직접 불러옵니다.

> GitHub Pages에서는 Python이나 MySQL을 직접 실행하지 않으므로, 데이터가 변경되면 이 노트북을 다시 실행해 JSON 파일을 갱신해야 합니다.
